# Backprop by Autograd

## Autograd Basics

In [ ]:
import torch

In [ ]:
x = torch.ones(1)
x.requires_grad

In [ ]:
y = torch.ones(1)
y.requires_grad

In [ ]:
z = x + y

In [ ]:
z.requires_grad

In [ ]:
x.requires_grad_()
x.requires_grad

In [ ]:
z = x + y

In [ ]:
z.requires_grad

## Backward

In [ ]:
y = torch.tensor([[1, 2], [3, 4]], dtype=torch.float32, requires_grad=True)
y

In [ ]:
z = 2*y + 1
z

In [ ]:
print("y.requires_grad :", y.requires_grad)
print("z.requires_grad :", z.requires_grad)

print("")

print("y.grad :", y.grad)
print("z.grad :", z.grad)

print("")

print("y.grad_fn :", y.grad_fn)
print("z.grad_fn :", z.grad_fn)

In [ ]:
gz = z.grad_fn
print("z.grad_fn :", gz)
print("Where is z.grad_fn?", id(gz))
print("Where is z.grad_fn in hex?", hex(id(gz)))

In [ ]:
# next_functions로 다음 backward 함수를 파악할 수 있음
# next_functions reveals the next backward op in the graph
z.grad_fn.next_functions

In [ ]:
# grad는 스칼라 출력에 대해서만 암묵적으로 생성됨
# grad can be implicitly created only for scalar outputs
z.backward()

In [ ]:
out = z.sum()
out

In [ ]:
print("y.requires_grad :", y.requires_grad)
print("z.requires_grad :", z.requires_grad)
print("out.requires_grad :", out.requires_grad)

print("")

print("y.grad :", y.grad)
print("z.grad :", z.grad)
print("out.grad :", out.grad)

print("")

print("y.grad_fn :", y.grad_fn)
print("z.grad_fn :", z.grad_fn)
print("out.grad_fn :", out.grad_fn)

In [ ]:
out.backward()

In [ ]:
print("y.requires_grad :", y.requires_grad)
print("z.requires_grad :", z.requires_grad)
print("out.requires_grad :", out.requires_grad)

print("")

print("y.grad :", y.grad)
print("z.grad :", z.grad)
print("out.grad :", out.grad)

print("")

print("y.grad_fn :", y.grad_fn)
print("z.grad_fn :", z.grad_fn)
print("out.grad_fn :", out.grad_fn)

In [ ]:
# 기본적으로 gradient는 leaf 변수에만 보존됨. 비-leaf 변수는
# By default, gradients are only retained for leaf variables. Non-leaf vars
# 비-leaf 변수의 gradient는 메모리 절약을 위해 보존되지 않음
# Non-leaf gradients are not retained — by design, to save memory
print("y == leaf? : ", y.is_leaf)
print("z == leaf? : ", z.is_leaf)

## Backward Twice

In [ ]:
y = torch.tensor([[1, 2], [3, 4]], dtype=torch.float32, requires_grad=True)
z = 2*y + 1
out = z.sum()
out.backward()
out.backward()

In [ ]:
y = torch.tensor([[1, 2], [3, 4]], dtype=torch.float32, requires_grad=True)
z = 2*y + 1
out = z.sum()
out.backward(retain_graph=True)
out.backward()

In [ ]:
y.grad

## .detach() vs .data

In [ ]:
# 그래프 위에 있는 변수는 바로 numpy로 변환할 수 없음
# A variable on the graph cannot be converted to numpy directly
z.numpy()

In [ ]:
# 따라서 .detach() 또는 .data 중 하나를 사용해야 함
# So we must use either .detach() or .data
z.detach().numpy()

In [ ]:
# 따라서 .detach() 또는 .data 중 하나를 사용해야 함
# So we must use either .detach() or .data
z.data.numpy()

@ .detach된 tensor는 requires_grad가 False이지만 기존 grad에 inplace 알려줌

In [ ]:
a = torch.tensor([1,2,3.], requires_grad = True)
b = a.exp()

# c는 b를 그래프에서 떼어낸 것으로 requires_grad=False
# c is detached from b's graph; requires_grad becomes False
# 그러나 원본 데이터는 공유됨
# But the underlying storage is shared
c = b.detach()
c.zero_()

In [ ]:
# c를 0으로 만들었으므로, 데이터를 공유하는 b도 0이 됨
# c was zeroed and shares storage with b, so b is also zeroed
b

In [ ]:
# exp 함수는 자기 자신의 출력값이 역전파에 필요
# exp() requires its own output during backprop
# 그러나 원본 데이터가 수정되었으므로 에러가 나는 것이 맞음
# But since the underlying data was modified, raising an error is the correct behavior
b.sum().backward()

In [ ]:
# backward가 진행되지 않았으므로 a.grad는 없어야 함
# Since backward did not run, a.grad must be None
a.grad

@ .data된 tensor는 grad에 변화를 줄 수 없음

In [ ]:
a = torch.tensor([1,2,3.], requires_grad = True)
b = a.exp()

# c를 0으로 초기화
# Zero-out c
c = b.data
c.zero_()

In [ ]:
# 마찬가지로 b도 0으로 초기화됨
# Likewise, b is also zeroed
b

In [ ]:
# inplace 변경이 있었음에도 에러가 나지 않음 (비정상 동작)
# In-place modification did NOT raise an error (incorrect behavior)
b.sum().backward()

In [ ]:
# 잘못된 결과
# Incorrect result
a.grad

방지하기 위해서는 detach().clone()을 사용

In [ ]:
a = torch.tensor([1,2,3.], requires_grad = True)
b = a.exp()

# c를 0으로 초기화
# Zero-out c
c = b.detach().clone()
c.zero_()

In [ ]:
b